## От данных к скалярной статистике: интуиция и вывод

---

### 1. Задача: зачем вообще диффузионные модели?

Пусть в каждый момент $t$ мы наблюдаем снапшот стакана $X_t \in \mathbb{R}^d$ — вектор высокой размерности. Мы хотим уметь отвечать на вопрос:

> «Это наблюдение типичное для нормального режима, или распределение изменилось?»

Наивный подход — оценить плотность $p_0(X_t)$ и смотреть, не упала ли она. Но в высоких измерениях $p_0$ в явном виде недоступна и плохо оценивается. Вместо этого диффузионные модели учат **градиент логарифма плотности** — score-функцию — и оказывается, что этого достаточно.

---

### 2. Прямой процесс: добавляем шум

Возьмём реальное наблюдение $x_0 \sim p_0$ и последовательно добавляем к нему гауссовский шум:

$$x_\tau = \mu(\tau)\, x_0 + \sigma(\tau)\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

где $\mu(\tau) = e^{-\frac{1}{2}\int_0^\tau \beta(s)\,ds}$ убывает, а $\sigma^2(\tau) = 1 - \mu^2(\tau)$ растёт. При $\tau \to T$ данные полностью «засыпаны» шумом: $x_T \approx \mathcal{N}(0, I)$ независимо от $x_0$.

**Ключевое наблюдение.** Зашумлённое $x_\tau$ можно записать как:

$$x_\tau = \mu(\tau)\, x_0 + \sigma(\tau)\, \varepsilon$$

Мы знаем, что зашумляли, и знаем каким шумом $\varepsilon$. Значит, теоретически можно восстановить $\varepsilon$ обратно — если знать, откуда пришёл $x_0$.

---

### 3. Что обучает диффузионная модель: денойзер

Обучаем нейросеть $\hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h})$, которая по зашумлённому наблюдению $x_\tau$ и контексту $\mathbf{h}$ предсказывает добавленный шум $\varepsilon$:

$$\mathcal{L}(\theta, \phi) = \mathbb{E}_{\tau,\, x_0,\, \varepsilon}\!\left[\left\|\hat\varepsilon_\theta\!\left(\mu(\tau)\, x_0 + \sigma(\tau)\,\varepsilon,\;\tau,\;\mathbf{h}\right) - \varepsilon\right\|^2\right]$$

Оптимальное решение этой задачи — условное математическое ожидание:

$$\hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h}) \;\xrightarrow{\text{обучение}}\; \mathbb{E}[\varepsilon \mid x_\tau, \mathbf{h}]$$

**Что это значит содержательно.** Сеть учится отвечать на вопрос: «Если зашумлённая точка выглядит вот так, то откуда скорее всего пришёл исходный $x_0$ и какой шум добавили?» Чем лучше сеть знает $p_0$, тем точнее это предсказание.

---

### 4. Связь денойзера и score: формула Твиди

**Теорема (Tweedie, 1956).** Для маргинальной плотности $p_\tau(x_\tau) = \int p(x_\tau \mid x_0)\, p_0(x_0)\, dx_0$:

$$\nabla_{x_\tau} \log p_\tau(x_\tau) = -\frac{\mathbb{E}[\varepsilon \mid x_\tau]}{\sigma(\tau)}$$

**Доказательство.** Берём $\nabla_{x_\tau} \log p_\tau$ и расписываем через закон полной вероятности:

$$\nabla_{x_\tau} \log p_\tau(x_\tau) = \frac{\nabla_{x_\tau} p_\tau}{p_\tau} = \int \underbrace{\nabla_{x_\tau} \log p(x_\tau \mid x_0)}_{= -(x_\tau - \mu(\tau) x_0)/\sigma^2(\tau)}\, p(x_0 \mid x_\tau)\, dx_0$$

$$= -\frac{x_\tau - \mu(\tau)\,\mathbb{E}[x_0 \mid x_\tau]}{\sigma^2(\tau)} = -\frac{\mathbb{E}[x_\tau - \mu(\tau) x_0 \mid x_\tau]}{\sigma^2(\tau)} = -\frac{\sigma(\tau)\,\mathbb{E}[\varepsilon \mid x_\tau]}{\sigma^2(\tau)} = -\frac{\mathbb{E}[\varepsilon \mid x_\tau]}{\sigma(\tau)} \qquad \square$$

**Следствие.** Обученный денойзер — это и есть score:

$$s_0(x_\tau, \tau, \mathbf{h}) \;=\; \nabla_{x_\tau} \log p_\tau(x_\tau \mid \mathbf{h}) \;=\; -\frac{\hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h})}{\sigma(\tau)}$$

Предсказывать шум и предсказывать score — **одно и то же**, только с разным нормировочным множителем.

---

### 5. Ошибка денойзинга как суррогат $-\log p_0$

Раскроем функцию потерь DSM через явный score matching:

$$\underbrace{\mathbb{E}_{x_0, \varepsilon}\!\left[\left\|\hat\varepsilon_\theta(x_\tau, \tau) - \varepsilon\right\|^2\right]}_{J_\text{DSM}} = \underbrace{\mathbb{E}_{p_\tau}\!\left[\left\|s_\theta(x_\tau) - \nabla_{x_\tau} \log p_\tau(x_\tau)\right\|^2\right]}_{J_\text{ESM}} + \text{const}$$

Оба критерия имеют одинаковый минимум. Из $J_\text{ESM}$ видно: чем меньше ошибка денойзинга, тем ближе $s_\theta$ к истинному score, тем точнее модель знает $p_0$.

Теперь из ELBO диффузионной модели:

$$\log p_0(x) \;\geq\; -\,\mathcal{L}_\text{DSM}(x, \mathbf{h}) + C$$

где $C$ не зависит от $x$. Следовательно:

$$\mathcal{L}_\text{DSM}(x, \mathbf{h}) \;\approx\; -\log p_0(x \mid \mathbf{h}) + C$$

**Интуиция.** Ошибка денойзинга — это оценка «неожиданности» наблюдения $x$ под моделью $p_0$.

- $x \sim p_0$: модель хорошо предсказывает шум $\Rightarrow$ **маленькая ошибка**
- $x \sim p_1 \neq p_0$: структура данных изменилась, предсказание хуже $\Rightarrow$ **ошибка растёт**

---

### 6. Откуда берётся скалярная статистика

Для каждого нового наблюдения $X_t$ выполняем следующее:

1. Сэмплируем шум: $\varepsilon \sim \mathcal{N}(0, I)$
2. Зашумляем при фиксированном $\tau$: $x_\tau = \mu(\tau)\, X_t + \sigma(\tau)\, \varepsilon$
3. Прогоняем денойзер: $\hat\varepsilon = \hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h}_{t-1})$
4. Вычисляем ошибку:

$$L_t = \left\|\hat\varepsilon_\theta\!\left(\mu(\tau)\, X_t + \sigma(\tau)\,\varepsilon,\;\tau,\;\mathbf{h}_{t-1}\right) - \varepsilon\right\|^2$$

Это скаляр. На hold-out выборке $\{X_1^{(0)}, \ldots, X_N^{(0)}\} \sim p_0$ калибруем:

$$\mu_0 = \frac{1}{N}\sum_{t=1}^N L_t^{(0)}, \qquad \sigma_0^2 = \frac{1}{N-1}\sum_{t=1}^N (L_t^{(0)} - \mu_0)^2$$

Нормируем:

$$\boxed{Z_t = \frac{L_t - \mu_0}{\sigma_0}}$$

---

### 7. Распределение $Z_t$ под $H_0$: почему $\mathcal{N}(0, 1)$

$L_t$ — это квадрат нормы вектора остатков $\hat\varepsilon_\theta(x_\tau) - \varepsilon$ в $\mathbb{R}^d$. Раскладываем:

$$L_t = \left\|\hat\varepsilon_\theta(x_\tau) - \varepsilon\right\|^2 = \sum_{j=1}^d \left(\hat\varepsilon_\theta^{(j)} - \varepsilon^{(j)}\right)^2$$

При хорошо обученном денойзере остатки $\varepsilon^{(j)} - \hat\varepsilon_\theta^{(j)}$ слабо зависимы и одинаково распределены. По **ЦПТ** при больших $d$:

$$L_t \;\xrightarrow{d}\; \mathcal{N}(\mu_0,\, \sigma_0^2)$$

После нормировки:

$$Z_t \mid \mathcal{F}_{t-1},\; X_t \sim p_0 \;\approx\; \mathcal{N}(0,\, 1)$$

На практике нормальность $Z_t$ проверяется непосредственно (Q-Q plot, тест Шапиро–Уилка) — это не предположение, а эмпирически верифицируемый факт.

---

### 8. Распределение $Z_t$ под $H_1$: почему сдвиг

**Утверждение.** При $X_t \sim p_1 \neq p_0$:

$$\mathbb{E}[L_t \mid H_1] = \mu_0 + \delta, \qquad \delta > 0$$

**Доказательство.** Запишем ошибку оптимального денойзера для $p_1$:

$$\mathbb{E}_{p_1}\!\left[\left\|\hat\varepsilon_\theta - \varepsilon\right\|^2\right] = \underbrace{\mathbb{E}_{p_1}\!\left[\left\|\hat\varepsilon_1 - \varepsilon\right\|^2\right]}_{\text{минимальная ошибка при } p_1} + \underbrace{\mathbb{E}_{p_1}\!\left[\left\|\hat\varepsilon_\theta - \hat\varepsilon_1\right\|^2\right]}_{\text{штраф за несоответствие модели}}$$

Второе слагаемое строго положительно при $p_1 \neq p_0$, потому что $\hat\varepsilon_\theta$ был обучен под $p_0$, а не под $p_1$. $\square$

При гауссовском приближении:

$$\boxed{Z_t \mid \mathcal{F}_{t-1},\; X_t \sim p_1 \;\approx\; \mathcal{N}(\delta,\, 1)}$$

---

### Итог: что мы получаем на выходе

На каждом шаге $t$ у нас есть скаляр $Z_t$ с простым и явно известным распределением:

$$Z_t \;\sim\; \begin{cases} \mathcal{N}(0,\; 1) & \text{нормальный режим} \\[4pt] \mathcal{N}(\delta,\; 1) & \text{после разладки} \end{cases}$$

Высокоразмерная задача обнаружения изменений в $p_0(x \mid \mathbf{h})$ над $\mathbb{R}^d$ **сведена к одномерной задаче** о сдвиге среднего в последовательности скаляров.

| | Нормальный режим | После разладки |
|--|--|--|
| $X_t$ | $\sim p_0(\cdot \mid \mathbf{h}_{t-1})$ | $\sim p_1(\cdot \mid \mathbf{h}_{t-1})$ |
| $L_t = \|\hat\varepsilon_\theta - \varepsilon\|^2$ | $\approx \mu_0$ | $\approx \mu_0 + \delta$ |
| $Z_t = (L_t - \mu_0)/\sigma_0$ | $\approx \mathcal{N}(0, 1)$ | $\approx \mathcal{N}(\delta, 1)$ |

Именно к этой последовательности $\{Z_t\}$ с известными распределениями гипотез применяется теория оптимального обнаружения разладок.


## Строгое обоснование нормальности $L_t$

---

### Постановка: $L_t$ как функция шума

Зафиксируем $x_0 \sim p_0$, контекст $\mathbf{h}$, уровень шума $\tau$. Сэмплируем $\varepsilon \sim \mathcal{N}(0, I_d)$ и пишем:

$$L_t = \|\hat\varepsilon_\theta(\underbrace{\mu(\tau) x_0 + \sigma(\tau)\varepsilon}_{=\, x_\tau},\; \tau,\; \mathbf{h}) - \varepsilon\|^2 = \sum_{j=1}^d e_j^2, \qquad e_j := \hat\varepsilon_\theta^{(j)}(x_\tau) - \varepsilon^{(j)}$$

Вопрос: при каких условиях $L_t \xrightarrow{d} \mathcal{N}$ при $d \to \infty$?

Сложность в том, что слагаемые $e_j^2$ **зависимы**: деноизер смешивает координаты через $x_\tau$, поэтому $e_j$ зависит от **всех** компонент $\varepsilon$ через $x_\tau^{(k)} = \mu x_0^{(k)} + \sigma \varepsilon^{(k)}$.

---

### Случай 1. Факторизованное $p_0$ — прямая теорема Линдеберга

Если $p_0$ имеет независимые координаты и деноизер **координатно-отделим** (т.е. $\hat\varepsilon_\theta^{(j)}$ зависит только от $x_\tau^{(j)}$), то:

$$e_j = \hat\varepsilon_\theta^{(j)}(x_\tau^{(j)}) - \varepsilon^{(j)}$$

Здесь $e_1, \ldots, e_d$ **независимы** (каждый зависит от своего $\varepsilon^{(j)}$).

**Теорема Линдеберга–Феллера.** Пусть $X_j = e_j^2$ независимы, $\mathbb{E}[X_j] = b_j$, $\mathrm{Var}(X_j) = v_j$, $s_d^2 = \sum_{j=1}^d v_j$. Тогда:

$$\frac{L_t - \sum_j b_j}{s_d} \xrightarrow{d} \mathcal{N}(0, 1)$$

если выполнено **условие Линдеберга**: для любого $\varepsilon_0 > 0$

$$\frac{1}{s_d^2} \sum_{j=1}^d \mathbb{E}\!\left[X_j^2\, \mathbf{1}_{|X_j| > \varepsilon_0 s_d}\right] \to 0$$

**Достаточное условие** (Ляпунов): $\exists\, \delta > 0$ такое, что $\sum_j \mathbb{E}[|X_j|^{2+\delta}] = o(s_d^{2+\delta})$.

В частности, если $b_j \geq b_{\min} > 0$ для всех $j$, то $s_d^2 \geq 2 b_{\min}^2 d$ и условие Ляпунова при $\delta = 1$ сводится к $\sum_j \mathbb{E}[e_j^6] = o(d^{3/2})$ — выполнено при ограниченных моментах. $\square$

---

### Случай 2. Линейный деноизер — квадратичная форма от гауссовского вектора

Рассмотрим линейный деноизер $\hat\varepsilon_\theta(x_\tau) = A x_\tau$. Тогда:

$$e = Ax_\tau - \varepsilon = \sigma A \varepsilon + \mu A x_0 - \varepsilon = \underbrace{(\sigma A - I)}_{=:\, B}\, \varepsilon + \underbrace{\mu A x_0}_{=:\, m}$$

$$L_t = \|B\varepsilon + m\|^2 = \varepsilon^\top B^\top B\, \varepsilon + 2m^\top B\, \varepsilon + \|m\|^2$$

Это **квадратичная форма** от $\varepsilon \sim \mathcal{N}(0, I_d)$. Применяем точный результат:

**Теорема (ЦПТ для квадратичных форм гауссовских векторов).** Пусть $Q = \varepsilon^\top M \varepsilon + a^\top \varepsilon$ c $\varepsilon \sim \mathcal{N}(0, I_d)$. Тогда:

$$\mathrm{Var}(Q) = 2\|M\|_F^2 + \|a\|^2$$

$$\frac{Q - \mathbb{E}[Q]}{\sqrt{\mathrm{Var}(Q)}} \xrightarrow{d} \mathcal{N}(0, 1) \iff \frac{\lambda_{\max}(M)^2}{\|M\|_F^2} \to 0$$

**Доказательство условия.** Пусть $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_d$ — собственные значения $M$. По спектральному разложению $Q = \sum_j \lambda_j z_j^2 + \tilde{a}^\top z$ где $z \sim \mathcal{N}(0, I)$. Характеристическая функция:

$$\varphi_{Q}(t) = \prod_{j=1}^d \frac{1}{\sqrt{1 - 2i\lambda_j t}}\, e^{i t \tilde a_j^2 / (1 - 2i\lambda_j t)}$$

Нормированная $\hat{Q} = (Q - \mathbb{E}[Q])/\sigma_Q$ сходится к $\mathcal{N}(0,1)$ по характеристическим функциям тогда и только тогда, когда $\max_j |\lambda_j| / \sigma_Q \to 0$. При $\|a\| = 0$: $\sigma_Q^2 = 2\|M\|_F^2$, и условие принимает вид:

$$\frac{\lambda_{\max}(M)^2}{\|M\|_F^2} = \frac{\lambda_{\max}^2}{\sum_j \lambda_j^2} \to 0 \qquad \square$$

**Это условие — именно условие на эффективный ранг матрицы $M = B^\top B$:**

$$\mathrm{eff\text{-}rank}(M) := \frac{\|M\|_F^2}{\lambda_{\max}^2(M)} = \frac{\sum_j \lambda_j^2}{\lambda_{\max}^2} \to \infty$$

**Геометрический смысл.** Деноизер "размазывает" ошибку по многим собственным направлениям, не концентрируя всё в одном. Если $M$ имеет $d$ равных собственных значений $\lambda$, то $\mathrm{eff\text{-}rank} = d \to \infty$ ✓. Если $M$ ранга $1$ с $\lambda_1 = \lambda$, то $\mathrm{eff\text{-}rank} = 1$ и CLT **не выполняется** — $L_t$ имеет распределение $\chi^2(1)$.

---

### Случай 3. Нелинейный деноизер — линеаризация через якобиан

Для гладкого нелинейного $\hat\varepsilon_\theta$ разложим по $\varepsilon$ в типичной точке $\varepsilon_0 = 0$:

$$\hat\varepsilon_\theta(\mu x_0 + \sigma \varepsilon) = \hat\varepsilon_\theta(\mu x_0) + \sigma J\, \varepsilon + O(\|\varepsilon\|^2 \cdot \|H\|)$$

где $J = \nabla_x \hat\varepsilon_\theta|_{x=\mu x_0} \in \mathbb{R}^{d \times d}$ — якобиан деноизера.

Вычитаем $\varepsilon$:

$$e = (\sigma J - I)\varepsilon + \underbrace{\hat\varepsilon_\theta(\mu x_0) - 0}_{=:\,m} + O(\|H\|\|\varepsilon\|^2)$$

При $d \to \infty$ остаток $O(\|H\|\|\varepsilon\|^2)$ вносит вклад $O(\|H\| \cdot d)$ в $L_t$, что меньше ведущего члена $O(d)$, если $\|H\|_{\mathrm{op}} = o(1)$ (типично для регуляризованных моделей).

В первом приближении:

$$L_t \approx \|(\sigma J - I)\varepsilon + m\|^2 \qquad \Longrightarrow \qquad \text{квадратичная форма с } M = (\sigma J - I)^\top(\sigma J - I)$$

**ЦПТ выполняется при:**

$$\boxed{\frac{\lambda_{\max}((\sigma J - I)^\top(\sigma J - I))}{\|(\sigma J - I)\|_F} = \frac{\|(\sigma J - I)\|_{\mathrm{op}}^2}{\|(\sigma J - I)\|_F} \to 0}$$

что эквивалентно $\mathrm{eff\text{-}rank}(J) \to \infty$, т.е. якобиан деноизера не вырожден в ранг-$1$.

---

### Итог: точная формулировка

**Теорема.** Пусть $\varepsilon \sim \mathcal{N}(0, I_d)$, деноизер $\hat\varepsilon_\theta$ дифференцируем с якобианом $J = \nabla_x \hat\varepsilon_\theta$ при $x = \mu x_0$. Если

$$\mathrm{eff\text{-}rank}(J) = \frac{\|J\|_F^2}{\|J\|_{\mathrm{op}}^2} \xrightarrow{d \to \infty} \infty$$

то:
$$\frac{L_t - \mathbb{E}[L_t]}{\sqrt{\mathrm{Var}(L_t)}} \xrightarrow{d} \mathcal{N}(0, 1)$$

**Что это значит практически.** Хорошо обученный UNet или трансформер имеет $\|J\|_F^2 = \Theta(d)$ (ошибка денойзинга расперделена по всем координатам) при $\|J\|_{\mathrm{op}} = O(1)$ (Lipschitz-константа модели ограничена), откуда $\mathrm{eff\text{-}rank}(J) = \Theta(d) \to \infty$.

**Когда CLT нарушается.** Деноизер с ранг-$1$ якобианом (например, предсказывает только вдоль одного направления в пространстве данных) даёт $L_t \sim \chi^2(1)$, а не нормальное распределение. На практике это проверяется Q-Q plot и тестом Шапиро–Уилка по калибровочным данным — не как допущение, а как верифицируемый факт.


## Гарантированная нормальность через $K$ независимых шумов

---

### Конструкция

Для каждого наблюдения $X_t$ сэмплируем $K$ независимых шумов и делаем $K$ прогонов через деноизер:

$$\varepsilon_1, \ldots, \varepsilon_K \overset{\mathrm{i.i.d.}}{\sim} \mathcal{N}(0, I_d), \qquad x_\tau^{(k)} = \mu(\tau)\, X_t + \sigma(\tau)\, \varepsilon_k$$

$$Y_k = \left\|\hat\varepsilon_\theta\!\left(x_\tau^{(k)},\, \tau,\, \mathbf{h}_{t-1}\right) - \varepsilon_k\right\|^2, \qquad k = 1, \ldots, K$$

Нормируем по калибровочным константам $\mu_0$, $\sigma_0$ и усредняем:

$$\boxed{Z_t^{(K)} = \frac{1}{\sqrt{K}} \sum_{k=1}^K \frac{Y_k - \mu_0}{\sigma_0}}$$

---

### Теорема о распределении

**Утверждение.** При любом деноизере $\hat\varepsilon_\theta$, любой размерности $d$ и любом распределении $p_0$:

$$Z_t^{(K)} \mid \mathcal{F}_{t-1},\; H_0 \;\xrightarrow{K \to \infty}\; \mathcal{N}(0, 1)$$

**Доказательство.** Зафиксируем $X_t$ и $\mathbf{h}_{t-1}$. Обозначим $\tilde Y_k = (Y_k - \mu_0)/\sigma_0$.

Ключевое наблюдение: $\tilde Y_1, \ldots, \tilde Y_K$ **независимы и одинаково распределены**, потому что $\varepsilon_k$ независимы и каждое $Y_k$ является функцией только своего $\varepsilon_k$:

$$Y_k = f(\varepsilon_k), \qquad f(\varepsilon) := \|\hat\varepsilon_\theta(\mu X_t + \sigma\varepsilon,\, \tau,\, \mathbf{h}_{t-1}) - \varepsilon\|^2$$

При $H_0$ (т.е. $X_t \sim p_0$) по определению калибровки:

$$\mathbb{E}[\tilde Y_k] = 0, \qquad \mathrm{Var}(\tilde Y_k) = 1$$

Тогда по **классической теореме Линдеберга–Леви** (i.i.d. ЦПТ):

$$Z_t^{(K)} = \frac{1}{\sqrt{K}} \sum_{k=1}^K \tilde Y_k \xrightarrow{K \to \infty} \mathcal{N}(0, 1) \qquad \square$$

Никаких условий на архитектуру, размерность данных или эффективный ранг якобиана не требуется.

---

### Распределение при $H_1$

После разладки $X_t \sim p_1 \neq p_0$, поэтому $\mathbb{E}[Y_k \mid H_1] = \mu_0 + \delta$ с $\delta > 0$. Тогда:

$$\mathbb{E}[\tilde Y_k \mid H_1] = \frac{\delta}{\sigma_0} =: \mu_1 > 0$$

По ЦПТ:

$$Z_t^{(K)} \mid H_1 \xrightarrow{K \to \infty} \mathcal{N}\!\left(\sqrt{K}\,\mu_1,\; 1\right)$$

Дрейф растёт как $\sqrt{K}$: **больше шумов — сильнее сигнал разладки**.

---

### Итог

| | $K = 1$ | $K \to \infty$ |
|---|---|---|
| Нормальность | эмпирический факт (зависит от eff-rank$(J)$) | **гарантирована теоремой** |
| Дрейф при $H_1$ | $\mu_1$ | $\sqrt{K}\,\mu_1$ |
| Прогонов модели на шаг | $1$ | $K$ |

Цена гарантии — $K$ прогонов деноизера вместо одного. На практике $K \in [10, 50]$ достаточно: при $K = 16$ дрейф усиливается в $4$ раза и нормальность $Z_t^{(K)}$ достигается с запасом.


## Можно ли обучить модель с $K$ шумами и имеет ли это смысл?

---

### Обучение: ничего не меняется

Модель $\hat\varepsilon_\theta$ обучается стандартным DSM с **одним** шумом за шаг:

$$\mathcal{L}(\theta, \phi) = \mathbb{E}_{t,\, \tau,\, \varepsilon}\!\left[\|\hat\varepsilon_\theta(\mu(\tau) X_t + \sigma(\tau)\varepsilon,\; \tau,\; \mathbf{h}_{t-1}) - \varepsilon\|^2\right]$$

$K$ шумов — это **модификация инференса**, а не обучения. Обученный деноизер используется как есть.

---

### Что на самом деле делают $K$ шумов

Зафиксируем наблюдение $X_t$. Определим **ожидаемую ошибку денойзинга** как функцию от $X_t$:

$$\bar{Y}(X_t) := \mathbb{E}_{\varepsilon}\!\left[\|\hat\varepsilon_\theta(\mu X_t + \sigma\varepsilon,\;\tau,\;\mathbf{h}_{t-1}) - \varepsilon\|^2\right]$$

Это детерминированная функция $X_t$ (усреднённая по всем возможным шумам). Она недоступна в явном виде для нелинейного деноизера, но легко оценивается Monte Carlo:

$$\frac{1}{K}\sum_{k=1}^K Y_k \xrightarrow{K \to \infty} \bar{Y}(X_t) \qquad \text{(ЗБЧЧ, т.к. }Y_k \text{ i.i.d. при фикс. }X_t\text{)}$$

**$K$ шумов — это не новая информация о распределении.** Это способ точнее оценить $\bar{Y}(X_t)$, устранив случайность от выбора конкретного $\varepsilon$.

---

### Что такое $\bar{Y}(X_t)$ теоретически

Для оптимально обученного деноизера по ELBO диффузионной модели:

$$\bar{Y}(X_t) = \mathbb{E}_{\tau,\,\varepsilon}\!\left[\|\hat\varepsilon_\theta(\mu X_t + \sigma\varepsilon,\,\tau) - \varepsilon\|^2\right] \approx -\log p_0(X_t \mid \mathbf{h}_{t-1}) + C$$

В пределе $K \to \infty$ наш тест-статистик сходится к:

$$\frac{1}{K}\sum_{k=1}^K Y_k - \mu_0 \;\xrightarrow{K\to\infty}\; \bar{Y}(X_t) - \mu_0 \;\approx\; -\log p_0(X_t) + C'$$

То есть мы оцениваем **log-likelihood наблюдения под $p_0$** — оптимальную статистику по лемме Неймана–Пирсона.

---

### Декомпозиция сигнала и шума

$$\underbrace{\frac{1}{K}\sum_{k=1}^K Y_k - \mu_0}_{\text{наблюдаемое}} = \underbrace{\bar{Y}(X_t) - \mu_0}_{\text{сигнал разладки}} + \underbrace{\frac{1}{K}\sum_{k=1}^K (Y_k - \bar{Y}(X_t))}_{\text{MC-шум} \sim O(1/\sqrt{K})}$$

| | $K = 1$ | $K \to \infty$ |
|---|---|---|
| Что наблюдаем | $Y_1 - \mu_0$ (зашумлено) | $\bar{Y}(X_t) - \mu_0$ (точно) |
| Откуда шум | MC-шум от $\varepsilon$ | исчезает |
| Что остаётся | сигнал + шум | чистый сигнал $\approx -\log p_0(X_t)$ |
| Источник информации | один $X_t$ | один $X_t$ |

**Важно:** увеличение $K$ не даёт новой информации о распределении — вся информация в $X_t$. $K$ лишь убирает случайность от выбора шума.

---

### Где сидит реальная информация о разладке

Рассмотрим последовательность сигналов:

$$\xi_t := \bar{Y}(X_t) - \mu_0$$

Под $H_0$ ($X_t \sim p_0$): $\quad\mathbb{E}_{X_t \sim p_0}[\xi_t] = 0$ по определению калибровки. ✓

Под $H_1$ ($X_t \sim p_1$): $\quad\mathbb{E}_{X_t \sim p_1}[\xi_t] = \underbrace{\mathbb{E}_{p_1}[\bar{Y}(X_t)] - \mu_0}_{=\,\delta > 0}$ — мощность зависит от расстояния $p_1$ до $p_0$.

Информация о разладке накапливается **по наблюдениям $X_t$**, а не по шумам $\varepsilon_k$. Шумы — это инструмент оценки $\bar{Y}$, не источник данных.

---

### Итого

Да, модель полностью обоснована:

- Обучение — стандартный DSM, без изменений
- В пределе $K \to \infty$ статистика сходится к $-\log p_0(X_t)$ — теоретически оптимальной
- При конечном $K$ — это MC-оценка оптимальной статистики с ошибкой $O(1/\sqrt{K})$
- Нормальность $Z_t^{(K)}$ гарантирована CLT при $K \geq 10{-}30$ на практике

Единственное ограничение: $K$-кратная вычислительная стоимость на каждый шаг инференса.


## Что даёт известная структура зашумления

Ядро VP-SDE известно точно:

$$p_\tau(x_\tau \mid x_0) = \mathcal{N}(x_\tau;\; \mu_\tau x_0,\; \sigma_\tau^2 I)$$

Это означает, что скор **маргинального** распределения $p_\tau^{(0)}(x_\tau) = \int p_\tau(x_\tau|x_0)\,p_0(x_0)\,dx_0$ выражается через денойзер по формуле Твиди:

$$\nabla_{x_\tau} \log p_\tau^{(0)}(x_\tau \mid \mathbf{h}) = -\frac{\hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h})}{\sigma_\tau}$$

Без известного ядра мы не могли бы связать предсказание $\hat\varepsilon_\theta$ со скором маргинала — именно гауссова форма ядра делает это возможным.

---

## Stein-статистика: точно нулевая под $H_0$

Из тождества Стейна для $p_\tau^{(0)}$: для **любой** тест-функции $\phi$

$$\mathbb{E}_{p_\tau^{(0)}}\!\left[s_\theta(x_\tau, \tau, \mathbf{h})^\top \phi(x_\tau) + \nabla_{x_\tau} \cdot \phi(x_\tau)\right] = 0$$

Определим **Stein score statistic** с каноническим выбором $\phi = s_\theta$:

$$\xi_t^{(\tau)} = \underbrace{\|s_\theta(x_\tau^{(t)}, \tau, \mathbf{h}_{t-1})\|^2}_{\text{норма скора}} + \underbrace{\nabla_{x_\tau} \cdot s_\theta(x_\tau^{(t)}, \tau, \mathbf{h}_{t-1})}_{\text{дивергенция скора}} = \frac{\|\hat\varepsilon_\theta\|^2}{\sigma_\tau^2} - \frac{\nabla_{x_\tau} \cdot \hat\varepsilon_\theta}{\sigma_\tau}$$

**Под $H_0$:** $\mathbb{E}[\xi_t^{(\tau)} \mid \mathcal{F}_{t-1}] = 0$ — **точно**, без CLT, без калибровки, для любого $p_0$.

**Под $H_1$:**

$$\mathbb{E}[\xi_t^{(\tau)} \mid \mathcal{F}_{t-1}] = \mathbb{E}_{p_1^{(\tau)}}\!\left[(s^{(0)} - s^{(1)})^\top s^{(0)}\right] \approx J_\tau(p_1 \,\|\, p_0)$$

где $J_\tau$ — **дивергенция Фишера** между $p_\tau^{(1)}$ и $p_\tau^{(0)}$:

$$J_\tau(p_1 \| p_0) = \mathbb{E}_{p_1^\tau}\!\left[\|s^{(0)}(x_\tau) - s^{(1)}(x_\tau)\|^2\right]$$

---

## Оптимальность: три уровня

### 1. Локальная оптимальность среди Stein-тестов

Среди всех статистик вида $\mathbb{E}_{p_1}[\mathcal{T}_{p_0^\tau} \phi]$, выбор $\phi = s_\theta^{(0)}$ максимизирует KSD:

$$\text{KSD}^2(p_1^\tau \| p_0^\tau) = \sup_{\|\phi\|_\mathcal{H} \leq 1} \left[\mathbb{E}_{p_1^\tau}\left[\mathcal{T}_{p_0^\tau}\phi\right]\right]^2$$

Выбор $\phi = s_\theta$ является **локально наиболее мощным** среди тестов, основанных на операторе Стейна.

### 2. Оптимальный уровень шума (неравенство обработки данных)

По неравенству обработки данных информация о разладке убывает при зашумлении:

$$I(x_0;\; p_0 \to p_1) \;\geq\; I(x_\tau;\; p_0 \to p_1) \quad \text{для всех } \tau > 0$$

Следствие: **меньший** $\tau$ сохраняет больше информации о разладке. Оптимально использовать наименьший $\tau$, при котором модель ещё точна.

### 3. Многомасштабная агрегация → Neyman–Pearson

ELBO диффузионной модели аппроксимирует лог-правдоподобие:

$$\log p_\theta(x \mid \mathbf{h}) \approx -\sum_\tau w_\tau \,\mathbb{E}_\varepsilon\|\hat\varepsilon_\theta - \varepsilon\|^2 + \text{const}$$

Лог-отношение правдоподобий $\log(p_1/p_0)$ — **оптимальный** тест по лемме Неймана–Пирсона. Поэтому CUSUM на многомасштабной статистике

$$\xi_t = \sum_{k=1}^K w_k \cdot \xi_t^{(\tau_k)}, \qquad w_k \propto \frac{1}{\sigma_{\tau_k}^2}$$

**асимптотически приближается к оптимальному тесту** по мере улучшения модели скора.

---

## Итоговая схема оптимальности



## Вывод

**Работающий подход:** CUSUM на Stein score statistic

$$\xi_t^{(\tau)} = \frac{\|\hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h}_{t-1})\|^2}{\sigma_\tau^2} - \frac{\nabla_{x_\tau} \cdot \hat\varepsilon_\theta(x_\tau, \tau, \mathbf{h}_{t-1})}{\sigma_\tau}, \qquad S_t = \left(S_{t-1} + \xi_t\right)^+$$

---

**Почему он работает без знания $p_0$ и $p_1$:**

Гауссова структура ядра зашумления $p_\tau(x_\tau \mid x_0) = \mathcal{N}(\mu_\tau x_0, \sigma_\tau^2 I)$ гарантирует через формулу Твиди, что предсказание денойзера $\hat\varepsilon_\theta$ есть скор маргинала $p_\tau^{(0)}$. Тождество Стейна для этого маргинала немедленно даёт:

$$\mathbb{E}\!\left[\xi_t^{(\tau)} \mid \mathcal{F}_{t-1},\; x_t \sim p_0\right] = 0 \quad \textbf{точно}$$

Это не CLT-приближение и не требует калибровки — это прямое следствие геометрии процесса зашумления.

---

**Почему он оптимален:**

| Уровень оптимальности | Результат |
|---|---|
| Среди Stein-тестов | Выбор $\phi = s_\theta$ максимизирует KSD → локально наиболее мощный |
| По уровню шума | Малый $\tau$ → меньше потери информации (неравенство обработки данных) |
| В классе последовательных тестов | CUSUM на $\xi_t$ минимизирует наихудшую задержку при фиксированном ARL (результат Лордена) |
| Асимптотически | Многомасштабная $\xi_t = \sum_k w_k \xi_t^{(\tau_k)}$ приближает лог-LR → граница Неймана–Пирсона |

---

Иными словами: известная структура зашумления превращает предсказание денойзера в **точно нулевой мартингал** под $H_0$, а стандартная теория CUSUM гарантирует оптимальность по задержке детекции. Никакого знания о $p_0$ или $p_1$ не требуется.
